# 02: Sell-Through Analytics Pipeline

**Objective**: Transform raw H&M transaction data into a **sell-through scorecard**
at the (product_type, colour_group) level, measuring:

- **Velocity**: Average weekly units sold  
- **Revenue intensity**: Revenue per SKU per week  
- **Week-over-week growth**: Acceleration/deceleration of recent sales  
- **Composite sell-through score**: Weighted blend for downstream fusion  

This scorecard feeds into Trend Fusion Engine as the structured
"demand signal" side of the equation.

---

## 2.1 Setup

In [ ]:
import sys, os, logging, warnings
warnings.filterwarnings("ignore")

IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = "/content/drive/MyDrive/FashionTrendAnalyzer"
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from src.config import DATA_PROCESSED, hm_config
from src.data_loader import load_processed, save_processed
from src.sell_through import (
    enrich_transactions,
    compute_velocity,
    compute_wow_growth,
    compute_seasonal_decomposition,
    build_sell_through_scorecard,
    classify_sell_through,
)
from src.viz import style_color_heatmap

sns.set_theme(style="whitegrid", palette="muted")
print("Setup complete.")

## 2.2 Load Processed Data

In [ ]:
articles = load_processed("articles_clean")
transactions = load_processed("transactions_sample")

# Restore datetime type if lost in parquet serialization
if not pd.api.types.is_datetime64_any_dtype(transactions["t_dat"]):
    transactions["t_dat"] = pd.to_datetime(transactions["t_dat"])

print(f"Articles: {articles.shape}")
print(f"Transactions: {transactions.shape}")
print(f"Date range: {transactions['t_dat'].min()} → {transactions['t_dat'].max()}")

## 2.3 Enrich Transactions with Article Attributes

Join transaction records with product metadata (type, color, section) so we can
analyze sell-through by style attributes.

In [ ]:
enriched = enrich_transactions(transactions, articles)
print(f"Enriched shape: {enriched.shape}")
print(f"New columns: {[c for c in enriched.columns if c not in transactions.columns]}")
enriched.head(3)

In [ ]:
# Verify join quality
null_product_type = enriched["product_type_name"].isna().sum()
print(f"Transactions without product type: {null_product_type:,} ({null_product_type/len(enriched):.2%})")
enriched = enriched.dropna(subset=["product_type_name"])
print(f"After dropping nulls: {len(enriched):,} transactions")

## 2.4 Sell-Through Velocity Analysis

In [ ]:
velocity = compute_velocity(enriched)
print(f"Velocity scorecard: {velocity.shape}")
velocity.head(10)

In [ ]:
# Top 20 highest-velocity product types
top_velocity = (
    velocity
    .groupby("product_type_name")
    .agg({"avg_weekly_units": "sum", "total_revenue": "sum"})
    .nlargest(20, "avg_weekly_units")
    .reset_index()
)

fig = px.bar(
    top_velocity, x="avg_weekly_units", y="product_type_name",
    orientation="h",
    title="Top 20 Product Types by Weekly Sell-Through Velocity",
    labels={"avg_weekly_units": "Avg Weekly Units", "product_type_name": ""},
    template="plotly_white",
    color="total_revenue",
    color_continuous_scale="Viridis",
)
fig.update_layout(height=500, width=800, yaxis=dict(autorange="reversed"))
fig.show()

## 2.5 Week-over-Week Growth

In [ ]:
growth = compute_wow_growth(enriched, trailing_weeks=8)
growth_by_type = (
    growth
    .groupby("product_type_name")["wow_growth_rate"]
    .mean()
    .sort_values(ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

growth_by_type.head(15).plot.barh(
    ax=axes[0], color="seagreen", title="Top 15 — Fastest Growing (WoW)"
)
axes[0].set_xlabel("WoW Growth Rate")

growth_by_type.tail(15).plot.barh(
    ax=axes[1], color="firebrick", title="Bottom 15 — Fastest Declining (WoW)"
)
axes[1].set_xlabel("WoW Growth Rate")

plt.tight_layout()
plt.show()

## 2.6 Seasonal Patterns

In [ ]:
seasonal = compute_seasonal_decomposition(enriched)

# Top 6 product types by volume for seasonal analysis
top6_types = (
    seasonal.groupby("product_type_name")["units_sold"]
    .sum()
    .nlargest(6)
    .index
)
seasonal_top = seasonal[seasonal["product_type_name"].isin(top6_types)]

fig = px.line(
    seasonal_top, x="year_month", y="units_sold",
    color="product_type_name",
    title="Seasonal Sales Patterns — Top 6 Product Types",
    labels={"year_month": "Month", "units_sold": "Units Sold", "product_type_name": "Product Type"},
    template="plotly_white",
)
fig.update_layout(height=400, width=900, hovermode="x unified")
fig.show()

## 2.7 Build Complete Sell-Through Scorecard

In [ ]:
scorecard = build_sell_through_scorecard(enriched)
scorecard = classify_sell_through(scorecard)

print(f"Scorecard shape: {scorecard.shape}")
print(f"\nClassification distribution:")
print(scorecard["sell_through_class"].value_counts())
scorecard.head(10)

In [ ]:
# Style-Color Heatmap
fig = style_color_heatmap(scorecard)
fig.show()

In [ ]:
# Rising Stars vs Fading Favorites scatter
fig = px.scatter(
    scorecard,
    x="velocity_score_normalized",
    y="wow_growth_rate",
    color="sell_through_class",
    size="total_revenue",
    hover_data=["product_type_name", "colour_group_name"],
    title="Sell-Through Classification: Velocity vs Growth",
    labels={
        "velocity_score_normalized": "Velocity Score (normalized)",
        "wow_growth_rate": "WoW Growth Rate",
    },
    template="plotly_white",
    color_discrete_map={
        "Rising Star": "#2ECC40",
        "Fading Favorite": "#FF4136",
        "Emerging": "#FF851B",
        "Dormant": "#AAAAAA",
    },
)
fig.update_layout(height=500, width=850)
fig.show()

## 2.8 Save Scorecard

In [ ]:
save_processed(scorecard, "sell_through_scorecard")
print(f"\nSaved sell_through_scorecard.parquet — {len(scorecard):,} rows")
print(f"Columns: {list(scorecard.columns)}")